In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("analysis").getOrCreate()

In [2]:
df = spark.read.parquet("cleaned_data")
df.show()

+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|          event_time|      event_type|product_id|        category_id|category_code|        brand|price|  user_id|        user_session| department|    event_timestamp|event_date|hour|
+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|2019-10-18 12:08:...|            view|   5730223|1487580007457883075|  accessories|Unknown_brand| 4.44|468808792|cbec98c5-792b-4e9...|accessories|2019-10-18 12:08:00|2019-10-18|  12|
|2019-10-18 12:08:...|            view|   5695491|1487580013581566154|  accessories|        estel|  2.7|169486644|ee44d6fd-7b81-420...|accessories|2019-10-18 12:08:00|2019-10-18|  12|
|2019-10-18 12:08:...|            view|   5809870|1487580009387261981|  accessor

In [3]:
df.describe().show()

+-------+--------------------+----------+------------------+--------------------+-------------------+-------------+------------------+--------------------+--------------------+-----------+-----------------+
|summary|          event_time|event_type|        product_id|         category_id|      category_code|        brand|             price|             user_id|        user_session| department|             hour|
+-------+--------------------+----------+------------------+--------------------+-------------------+-------------+------------------+--------------------+--------------------+-----------+-----------------+
|  count|             4095560|   4095560|           4095560|             4095560|            4095560|      4095560|           4095560|             4095560|             4095560|    4095560|          4095560|
|   mean|                NULL|      NULL| 5467826.461246326|1.545583709356684...|               NULL|         NULL|   8.5478179345454|5.0135213905275345E8|                N

In [4]:
from pyspark.sql import functions as F

df.select(F.min("event_date")).show()

+---------------+
|min(event_date)|
+---------------+
|     2019-10-01|
+---------------+



In [5]:
# Peak traffic on website
df.groupBy("hour").count().orderBy("count", ascending=False).show(1)

+----+------+
|hour| count|
+----+------+
|  19|264054|
+----+------+
only showing top 1 row


In [6]:
# Let's find which brand product customers added in their cart more
df.filter((df["event_type"] == "cart") & (df["brand"] != "Unknown_brand")).groupBy("brand").count().orderBy("count", ascending=False).show(3)

+------+-----+
| brand|count|
+------+-----+
|runail|98695|
| irisk|78240|
|masura|58651|
+------+-----+
only showing top 3 rows


### Let's find the conversion rate of brands

In [7]:
# 1. Get total unique shoppers per brand (View or Cart)
total_shoppers = df.filter(F.col("event_type").isin("view", "cart")) \
    .groupBy("brand") \
    .agg(F.countDistinct("user_id").alias("total_interested_users"))

In [8]:
# 2. Get total unique buyers per brand
total_buyers = df.filter(F.col("event_type") == "purchase") \
    .groupBy("brand") \
    .agg(F.countDistinct("user_id").alias("total_buyers"))

In [9]:
# 3. Join them and calculate the "True" Rate
final_conversion = total_shoppers.join(total_buyers, "brand", "left").fillna(0)

final_conversion = final_conversion.withColumn(
    "true_conversion_rate",
    (F.col("total_buyers") / F.col("total_interested_users")) * 100
)

final_conversion.orderBy("true_conversion_rate", ascending=False).show(10)

+---------+----------------------+------------+--------------------+
|    brand|total_interested_users|total_buyers|true_conversion_rate|
+---------+----------------------+------------+--------------------+
|   cosima|                    40|          11|  27.500000000000004|
| severina|                  5288|        1419|  26.834341906202724|
|   eunyul|                   588|         142|  24.149659863945576|
|    smart|                  1874|         441|  23.532550693703307|
|  nitrile|                   364|          83|  22.802197802197803|
| supertan|                   162|          36|   22.22222222222222|
| nitrimax|                  1466|         323|  22.032742155525238|
|   elskin|                   508|         108|   21.25984251968504|
|   dermal|                   548|         115|  20.985401459854014|
|bpw.style|                 14059|        2698|   19.19055409346326|
+---------+----------------------+------------+--------------------+
only showing top 10 rows


### Now let's find out revenue by department

In [10]:
df.filter(df["event_type"] == "purchase").groupBy("department").sum("price").orderBy("sum(price)", ascending=False).show()

+-----------+------------------+
| department|        sum(price)|
+-----------+------------------+
|accessories|1163751.9099999801|
| appliances|          28789.57|
|  furniture|          10894.73|
|    apparel|           5502.83|
| stationery|3154.9499999999975|
+-----------+------------------+

